# Stokes drift vs Eulerian currents in the southern Kiel Bight

Comparing wave-driven Stokes drift against Eulerian surface currents
to understand the wave contribution to near-surface drift. Stokes drift
is large at the surface but decays exponentially with depth, so it
matters most for shallow drogues. We look at the deployment period
2023-04-24 to 2023-04-27 in the southern Kiel Bight.

## Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

## Parameters

In [ ]:
LON_START = 9.75
LON_END = 11.0
LAT_START = 54.3
LAT_END = 54.55
TIME_START = "2023-04-24"
TIME_END = "2023-04-27"
CMEMS_DIR = "data/cmems"

## Load wave data

CMEMS Baltic wave analysis (hourly, native grid). We take the surface
Stokes drift components, significant wave height, and peak period.
Coordinates are regularized with linspace so that streamplot works.

The Stokes drift at 3 m is estimated using the deep-water monochromatic
approximation: $u_{St}(z) = u_{St}(0) \exp(2kz)$ with $k = \sigma^2/g$.
This overestimates $k$ (underestimates wavelength) in the shallow Kiel
Bight where intermediate-depth dispersion applies. The estimated 3 m
Stokes drift is therefore a rough upper bound.

In [ ]:
LON = slice(LON_START, LON_END)
LAT = slice(LAT_START, LAT_END)
TIME = slice(TIME_START, TIME_END)

ds_wav = xr.open_dataset(CMEMS_DIR + "/cmems_mod_bal_wav_anfc_PT1H-i.nc").sel(
    longitude=LON, latitude=LAT, time=TIME,
)[
    ["VSDX", "VSDY", "VHM0", "VTPK", "VHM0_WW", "VTM01_WW", "VHM0_SW1", "VTM01_SW1"]
].load()

ds_wav = ds_wav.assign_coords(
    longitude=np.linspace(float(ds_wav.longitude[0]), float(ds_wav.longitude[-1]), len(ds_wav.longitude)),
    latitude=np.linspace(float(ds_wav.latitude[0]), float(ds_wav.latitude[-1]), len(ds_wav.latitude)),
)

ds_wav["stokes_speed"] = np.sqrt(ds_wav["VSDX"]**2 + ds_wav["VSDY"]**2)

# Estimated Stokes at 3m depth using deep water approximation:
# k = (2*pi)^2 / (g * Tp^2), then u_St(z) = u_St(0) * exp(-2*k*z)
g = 9.81
k = (2 * np.pi) ** 2 / (g * ds_wav["VTPK"] ** 2)
ds_wav["stokes_speed_3m"] = ds_wav["stokes_speed"] * np.exp(-2 * k * 3.0)

ds_wav

## Load current data

CMEMS Baltic physics analysis (hourly). We extract Eulerian currents
at three depths: surface, 1.5 m, and 3 m (using nearest-neighbor
selection on the depth coordinate).

In [ ]:
ds_phy_full = xr.open_dataset(CMEMS_DIR + "/cmems_mod_bal_phy_anfc_PT1H-i.nc").sel(
    longitude=LON, latitude=LAT, time=TIME,
)

depths = {"0m": 0, "1.5m": 1.5, "3m": 3}
ds_phy = {}
for label, z in depths.items():
    d = ds_phy_full.sel(depth=z, method="nearest")[["uo", "vo"]].load()
    d = d.assign_coords(
        longitude=np.linspace(float(d.longitude[0]), float(d.longitude[-1]), len(d.longitude)),
        latitude=np.linspace(float(d.latitude[0]), float(d.latitude[-1]), len(d.latitude)),
    )
    d["speed"] = np.sqrt(d["uo"] ** 2 + d["vo"] ** 2)
    ds_phy[label] = d
    print(f"  {label}: actual depth = {float(d.depth):.2f} m")

## Mean fields

2x2 panel: mean Stokes drift (surface, with streamlines) and mean
Eulerian currents at 0 m, 1.5 m, 3 m. All panels share a unified
color scale set by the 95th percentile across all four fields.

In [ ]:
# Compute mean velocity fields
stokes_mean = ds_wav[["VSDX", "VSDY"]].mean("time")
stokes_speed_mean = np.sqrt(stokes_mean["VSDX"] ** 2 + stokes_mean["VSDY"] ** 2)

current_mean = {label: d[["uo", "vo"]].mean("time") for label, d in ds_phy.items()}
current_speed_mean = {label: np.sqrt(m["uo"] ** 2 + m["vo"] ** 2) for label, m in current_mean.items()}

# Unified color scale from P95 of all mean speed fields
all_mean = np.concatenate(
    [stokes_speed_mean.values.ravel()]
    + [s.values.ravel() for s in current_speed_mean.values()]
)
vmax_mean = float(np.nanpercentile(all_mean, 95))

fig, axes = plt.subplots(2, 2)

stokes_speed_mean.plot(ax=axes[0, 0], vmin=0, vmax=vmax_mean)
stokes_mean.plot.streamplot(x="longitude", y="latitude", u="VSDX", v="VSDY", ax=axes[0, 0])
axes[0, 0].set_title("Mean Stokes drift (surface)")

panel_map = {"0m": axes[0, 1], "1.5m": axes[1, 0], "3m": axes[1, 1]}
for label in depths:
    ax = panel_map[label]
    current_speed_mean[label].plot(ax=ax, vmin=0, vmax=vmax_mean)
    current_mean[label].plot.streamplot(x="longitude", y="latitude", u="uo", v="vo", ax=ax)
    ax.set_title(f"Mean current (z = {label})")

plt.tight_layout()
plt.show()

## RMS fields

Same layout: RMS Stokes drift and RMS currents at three depths, unified
color scale.

In [ ]:
stokes_rms = np.sqrt((ds_wav["stokes_speed"] ** 2).mean("time"))
current_rms = {label: np.sqrt((d["speed"] ** 2).mean("time")) for label, d in ds_phy.items()}

all_rms = np.concatenate(
    [stokes_rms.values.ravel()] + [r.values.ravel() for r in current_rms.values()]
)
vmax_rms = float(np.nanpercentile(all_rms, 95))

fig, axes = plt.subplots(2, 2)

stokes_rms.plot(ax=axes[0, 0], vmin=0, vmax=vmax_rms)
axes[0, 0].set_title("RMS Stokes drift (surface)")

panel_map = {"0m": axes[0, 1], "1.5m": axes[1, 0], "3m": axes[1, 1]}
for label in depths:
    ax = panel_map[label]
    current_rms[label].plot(ax=ax, vmin=0, vmax=vmax_rms)
    ax.set_title(f"RMS current (z = {label})")

plt.tight_layout()
plt.show()

## Time series

Spatial median speed across the region vs time. Blue lines are Stokes
drift (solid = surface, dashed = estimated 3 m), red lines are Eulerian
currents (solid = 0 m, dashed = 1.5 m, dotted = 3 m).

In [ ]:
fig, ax = plt.subplots()

# Stokes: blue
stokes_median = ds_wav["stokes_speed"].median(["longitude", "latitude"])
stokes_3m_median = ds_wav["stokes_speed_3m"].median(["longitude", "latitude"])
ax.plot(stokes_median.time, stokes_median, color="tab:blue", linestyle="-", label="Stokes (surface)")
ax.plot(stokes_3m_median.time, stokes_3m_median, color="tab:blue", linestyle="--", label="Stokes (3 m, est.)")

# Currents: red
depth_styles = {"0m": "-", "1.5m": "--", "3m": ":"}
for label, d in ds_phy.items():
    median_ts = d["speed"].median(["longitude", "latitude"])
    ax.plot(median_ts.time, median_ts, color="tab:red", linestyle=depth_styles[label], label=f"Current (z = {label})")

ax.set_ylabel("Speed [m/s]")
ax.set_title("Spatial median speed across southern Kiel Bight")
ax.legend()
plt.show()

## Wave partitions

The CMEMS Baltic wave model provides partitioned wave parameters: wind
waves (WW) and primary swell (SW1). In the semi-enclosed Baltic, we
expect wind waves to dominate over swell.

In [ ]:
fig, axes = plt.subplots(1, 2)

# Significant wave height: wind waves vs swell
hs_ww = ds_wav["VHM0_WW"].median(["longitude", "latitude"])
hs_sw = ds_wav["VHM0_SW1"].median(["longitude", "latitude"])
axes[0].plot(hs_ww.time, hs_ww, label="Wind waves (WW)")
axes[0].plot(hs_sw.time, hs_sw, label="Primary swell (SW1)")
axes[0].set_ylabel("Hs [m]")
axes[0].set_title("Significant wave height by partition")
axes[0].legend()

# Mean period: wind waves vs swell
tm_ww = ds_wav["VTM01_WW"].median(["longitude", "latitude"])
tm_sw = ds_wav["VTM01_SW1"].median(["longitude", "latitude"])
axes[1].plot(tm_ww.time, tm_ww, label="Wind waves (WW)")
axes[1].plot(tm_sw.time, tm_sw, label="Primary swell (SW1)")
axes[1].set_ylabel("Tm01 [s]")
axes[1].set_title("Mean wave period by partition")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Mean Hs_WW: {float(ds_wav['VHM0_WW'].mean()):.3f} m")
print(f"Mean Hs_SW1: {float(ds_wav['VHM0_SW1'].mean()):.3f} m")
print(f"Ratio Hs_WW / Hs_total: {float(ds_wav['VHM0_WW'].mean() / ds_wav['VHM0'].mean()):.2f}")

## Summary statistics

Comparison of Stokes drift and Eulerian current speed at different
depths, averaged over the region and the deployment period.

In [9]:
fields = {
    "Stokes 0m": ds_wav["stokes_speed"],
    "Stokes 3m": ds_wav["stokes_speed_3m"],
    "Curr 0m": ds_phy["0m"]["speed"],
    "Curr 1.5m": ds_phy["1.5m"]["speed"],
    "Curr 3m": ds_phy["3m"]["speed"],
}

print(f"Southern Kiel Bight, {TIME.start} to {TIME.stop}")
print(f"{'':>12s} {'Stokes 0m':>10s} {'Stokes 3m':>10s} {'Curr 0m':>10s} {'Curr 1.5m':>10s} {'Curr 3m':>10s}")

for stat, func in [
    ("Mean", lambda x: float(x.mean())),
    ("RMS", lambda x: float(np.sqrt((x**2).mean()))),
    ("Max", lambda x: float(x.max())),
    ("P95", lambda x: float(x.quantile(0.95))),
]:
    vals = [func(f) for f in fields.values()]
    print(f"{stat:>12s}" + "".join(f"{v:10.4f}" for v in vals))

# Stokes as fraction of current
stokes_mean = float(ds_wav["stokes_speed"].mean())
stokes_3m_mean = float(ds_wav["stokes_speed_3m"].mean())
curr_0m_mean = float(ds_phy["0m"]["speed"].mean())
curr_3m_mean = float(ds_phy["3m"]["speed"].mean())

print(f"\nStokes / Current ratio:")
print(f"  Surface: {stokes_mean / curr_0m_mean:.0%}")
print(f"  At 3 m:  {stokes_3m_mean / curr_3m_mean:.0%}")

Southern Kiel Bight, 2023-04-24 to 2023-04-27
              Stokes 0m  Stokes 3m    Curr 0m  Curr 1.5m    Curr 3m
        Mean    0.1176    0.0265    0.0905    0.0896    0.0880
         RMS    0.1294    0.0375    0.1128    0.1120    0.1103
         Max    0.2434    0.1167    0.4375    0.4347    0.4321
         P95    0.2059    0.0833    0.2272    0.2257    0.2229

Stokes / Current ratio:
  Surface: 130%
  At 3 m:  30%


## Summary

<!-- TODO: fill in after running the notebook -->